# P2 · N0 — Environment, Data Check, and Splits

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

What this notebook does:

1. Mounts Drive and imports the `llmauth_*` modules
2. Finds your four data files and records their SHA-256 hashes
3. Loads each corpus **safely** (see the `None` warning in §5)
4. Builds the 20% authoring / 80% evaluation split for each corpus

**Resumability.** Every expensive step is checkpointed to Drive. If Colab
disconnects, re-run the notebook top to bottom — completed steps are skipped.
To redo a step, delete its file from `checkpoints/` and re-run.

**Self-healing.** Every cell rebuilds whatever it needs. If the runtime
restarts partway through, just re-run from the top — no cell depends on a
variable silently left behind by an earlier one.

**Runtime:** a few minutes. No GPU. No API keys.

## 1 · Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Setup — paths, imports, helpers

This cell defines everything the rest of the notebook uses. It is safe to
re-run at any point; that is the fix if you ever hit a `NameError`.

Adjust `ROOT` only if you renamed the project folder, and `MODULES` if you
moved the `llmauth_*.py` files.

In [ ]:
import sys, os, json, hashlib, datetime
from pathlib import Path

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'          # where the llmauth_*.py files live
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'

assert ROOT.exists(),    f'project folder not found: {ROOT}'
assert MODULES.exists(), f'modules folder not found: {MODULES}'
for d in (CHECKPOINTS, ARTIFACTS):
    d.mkdir(parents=True, exist_ok=True)

if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import llmauth_prompts as P
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)

# --- helpers (idempotent: call these instead of relying on earlier cells) ---

# filename fragments -> corpus id. Edit if your filenames differ.
PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}
SUBSAMPLE = {'nyc_tlc_yellow': 1_000_000}   # C4 only

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for blk in iter(lambda: fh.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

def discover_data():
    """Search the project folder for the four corpora. Returns {corpus: Path}."""
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv', '.parquet', '.xlsx')]
    found = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)  # prefer largest
        if hits:
            found[corpus] = hits[0]
    return found

def load_all(data_paths):
    """Load every discovered corpus using its registered read config."""
    return {c: C.load_corpus(c, p)[0] for c, p in data_paths.items()}

print('census    ', C.CENSUS_VERSION)
print('prompts   ', P.PROMPT_VERSION)
print('checkpoint', CK.CHECKPOINT_VERSION)
print('modules   ', MODULES)
print('registered corpora:', list(C.CORPUS_READ_CONFIG))

## 3 · Find the data files

Searches the whole project folder rather than assuming a layout. If a corpus
shows `MISSING`, upload it or edit `PATTERNS` in §2 and re-run.

In [ ]:
DATA_PATHS = discover_data()

print(f'{"corpus":<18} {"status":<9} {"size":>14}  file')
print('-' * 80)
for corpus in PATTERNS:
    p = DATA_PATHS.get(corpus)
    if p:
        print(f'{corpus:<18} {"FOUND":<9} {p.stat().st_size:>14,}  {p.relative_to(ROOT)}')
    else:
        print(f'{corpus:<18} {"MISSING":<9} {"-":>14}  --')

missing = [c for c in PATTERNS if c not in DATA_PATHS]
if missing:
    print(f'\n!! Missing: {missing}')
    print('   Upload them, or edit PATTERNS in section 2 to match your filenames.')
else:
    print('\nAll four corpora found.')

## 4 · Record file hashes

These go in the reproducibility artifact. If a source file is ever replaced
upstream, these hashes prove which version you used.

In [ ]:
DATA_PATHS = globals().get('DATA_PATHS') or discover_data()

def build_hashes():
    return {c: {'path': str(p.relative_to(ROOT)),
                'bytes': p.stat().st_size,
                'sha256': sha256_file(p)}
            for c, p in sorted(DATA_PATHS.items())}

file_hashes, _ = ckpt.step('source_file_hashes', 'json', build_hashes,
                           code_version=C.CENSUS_VERSION)

for c, h in file_hashes.items():
    print(f'{c:<18} {h["sha256"][:16]}...  {h["bytes"]:>14,} bytes')

## 5 · Load each corpus safely

`load_corpus` uses an explicit per-corpus read configuration and **refuses to
guess**.

> **Why this matters.** pandas' default NA list contains the string `"None"`.
> A plain `pd.read_csv` on `diabetic_data.csv` turns all 96,420 occurrences of
> `max_glu_serum = "None"` into `NaN`. That value is *valid* — it means the lab
> test was not administered — and losing it would silently destroy one of this
> paper's central findings. Never call `pd.read_csv` on these files directly.

The assertions below hard-fail if any sentinel was lost on load.

In [ ]:
DATA_PATHS = globals().get('DATA_PATHS') or discover_data()
corpora = load_all(DATA_PATHS)

for c, df in corpora.items():
    print(f'{c:<18} {df.shape[0]:>9,} rows x {df.shape[1]:>2} cols')

print()
if 'diabetes_130us' in corpora:
    d = corpora['diabetes_130us']
    assert (d.max_glu_serum == 'None').sum() == 96420, "the 'None' sentinel was destroyed on load"
    assert d.max_glu_serum.isna().sum() == 0
    assert (d.weight == '?').sum() == 98569
    print("OK  diabetes: 'None' (96,420) and '?' (98,569) sentinels intact")

if 'bank_marketing' in corpora:
    b = corpora['bank_marketing']
    assert (b.pdays == -1).sum() == 36954, 'pdays sentinel count unexpected'
    assert (b.poutcome == 'unknown').sum() == 36959
    print("OK  bank: pdays=-1 (36,954) and poutcome='unknown' (36,959) intact")

if 'nyc_tlc_yellow' in corpora:
    t = corpora['nyc_tlc_yellow']
    n99, nnull = int((t.RatecodeID == 99).sum()), int(t.RatecodeID.isna().sum())
    assert n99 > 0, 'RatecodeID=99 sentinel absent — wrong file?'
    print(f"OK  tlc: RatecodeID=99 ({n99:,}) and nulls ({nnull:,}) both present")

if 'online_retail_ii' in corpora:
    print(f"OK  retail: {len(corpora['online_retail_ii']):,} rows loaded")

## 6 · Build the authoring / evaluation splits

Rules are authored from a profile of the **20% authoring split**. Every metric
is computed on the disjoint **80% evaluation split**.

Without this separation the census would describe the same rows the rules are
later scored on — leakage a reviewer will find. C4 is subsampled to 1M rows
first, with a pinned seed. Splits are deterministic: same seed, same rows.

In [ ]:
if 'corpora' not in globals():
    DATA_PATHS = globals().get('DATA_PATHS') or discover_data()
    corpora = load_all(DATA_PATHS)

splits, manifests = {}, {}
for corpus, df in corpora.items():
    def build(corpus=corpus, df=df):
        auth, ev, man = C.make_split(df, corpus,
                                     subsample_rows=SUBSAMPLE.get(corpus))
        return {'manifest': man.__dict__,
                'authoring_index': auth.index.tolist(),
                'evaluation_index': ev.index.tolist()}
    payload, _ = ckpt.step(f'split_{corpus}', 'json', build,
                           code_version=C.CENSUS_VERSION)
    manifests[corpus] = payload['manifest']
    splits[corpus] = (df.loc[payload['authoring_index']],
                      df.loc[payload['evaluation_index']])

print()
print(f'{"corpus":<18} {"authoring":>11} {"evaluation":>11}  authoring index sha')
print('-' * 70)
for corpus, m in manifests.items():
    print(f'{corpus:<18} {m["authoring_rows"]:>11,} {m["evaluation_rows"]:>11,}  '
          f'{m["authoring_index_sha256"][:16]}...')

## 7 · Verify the splits

In [ ]:
ok = True
for corpus, (auth, ev) in splits.items():
    overlap = len(set(auth.index) & set(ev.index))
    total   = len(auth) + len(ev)
    frac    = len(auth) / total
    source  = manifests[corpus]['subsample_rows'] or manifests[corpus]['source_rows']
    good = (overlap == 0) and (total == source) and abs(frac - 0.20) < 0.001
    ok &= good
    print(f'{"OK " if good else "!! "}{corpus:<18} overlap={overlap}  '
          f'total={total:,}/{source:,}  authoring={frac:.4f}')

print('\nAll splits valid.' if ok else '\n!! SPLIT PROBLEM — do not proceed to N1.')

## 8 · Write the provenance record

In [ ]:
def build_provenance():
    return {
        'notebook': 'P2_N0_environment',
        'generated_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
        'census_version': C.CENSUS_VERSION,
        'prompt_version': P.PROMPT_VERSION,
        'checkpoint_version': CK.CHECKPOINT_VERSION,
        'source_files': file_hashes,
        'read_config': {c: C.CORPUS_READ_CONFIG[c] for c in DATA_PATHS},
        'splits': manifests,
    }

prov, _ = ckpt.step('n0_provenance', 'json', build_provenance,
                    code_version=C.CENSUS_VERSION)

out = ARTIFACTS / 'N0_provenance.json'
out.write_text(json.dumps(prov, indent=2, default=str))
print('wrote', out)

## 9 · Status

Re-run this cell any time to see what is already built.

**To rebuild a step:** delete its `.json` file from `checkpoints/` and re-run
the cell that creates it.

In [ ]:
ckpt.status()
print()
print('N0 complete. Next: N1 — census and A2-A5 prompt payloads.')